# Electricity Mix Calibration — Morocco Baseline

**Goal:** Take the original `df_input_1.csv` and produce a baseline with the following constant electricity generation mix for the full 2015–2070 horizon:

| Technology | MSP |
|---|---|
| `pp_coal` | 0.62 |
| `pp_wind` | 0.14 |
| `pp_gas` | 0.10 |
| `pp_oil` | 0.04 |
| `pp_solar` | 0.04 |
| `pp_nuclear` | 0.01 |
| **Σ MSP** | **0.95** |

The remaining 5% is left free for NemoMod's LP to allocate (typically absorbed by `pp_hydropower` thanks to its ~2.6 GW residual capacity).

**In addition**, we apply the feasibility tweaks required so the `TX:ENTC:TARGET_RENEWABLE_ELEC` transformation (95 % renewables by 2050) does not break NemoMod when applied downstream:

1. Enable `pp_nuclear` (original input has `max_capacity = 0` → infeasible with MSP > 0).
2. Loosen `total_annual_max_capacity_investment_pp_wind_gw` (0.2 → −999) so the renewable ramp can scale wind.
3. Loosen `total_annual_max_capacity_investment_pp_biomass_gw` (0 → −999) so biomass can expand in the strategy.

**Output:** `df_input_baseline_mix.csv` written next to the original input.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_DIR = Path('/Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data')
SRC = INPUT_DIR / 'df_input_1.csv'
OUT = INPUT_DIR / 'df_input_baseline_mix.csv'

df = pd.read_csv(SRC)
print(f'shape: {df.shape}')
print(f'time_period: {df.time_period.min()}..{df.time_period.max()}  (2015..{2015 + df.time_period.max()})')

shape: (56, 2442)
time_period: 0..55  (2015..2070)


## 1. Target mix and helpers

In [2]:
MSP_TARGET = {
    'pp_coal':    0.62,
    'pp_wind':    0.14,
    'pp_gas':     0.10,
    'pp_oil':     0.04,
    'pp_solar':   0.04,
    'pp_nuclear': 0.01,
    # remaining techs (biogas, biomass, coal_ccs, gas_ccs, geothermal, hydropower, ocean, waste_incineration) = 0
}
assert abs(sum(MSP_TARGET.values()) - 0.95) < 1e-9, f'Σ MSP = {sum(MSP_TARGET.values())} (must be 0.95)'

ALL_PP = [
    'pp_biogas', 'pp_biomass', 'pp_coal', 'pp_coal_ccs', 'pp_gas', 'pp_gas_ccs',
    'pp_geothermal', 'pp_hydropower', 'pp_nuclear', 'pp_ocean', 'pp_oil',
    'pp_solar', 'pp_waste_incineration', 'pp_wind',
]

def msp_col(t): return f'nemomod_entc_frac_min_share_production_{t}'
def inv_col(t): return f'nemomod_entc_total_annual_max_capacity_investment_{t}_gw'
def cap_col(t): return f'nemomod_entc_total_annual_max_capacity_{t}_gw'
def mpi_col(t): return f'frac_entc_max_elec_production_increase_to_satisfy_msp_{t}'

## 2. Diagnose the original input
Inspect MSP, residual capacity and caps to see the starting point.

In [3]:
diag = []
for t in ALL_PP:
    row = {'tech': t}
    row['msp_t0']     = df[msp_col(t)].iloc[0] if msp_col(t) in df.columns else None
    row['res_cap_t0'] = df[f'nemomod_entc_residual_capacity_{t}_gw'].iloc[0] if f'nemomod_entc_residual_capacity_{t}_gw' in df.columns else None
    row['max_cap']    = df[cap_col(t)].unique()[0] if cap_col(t) in df.columns else None
    row['max_inv']    = df[inv_col(t)].unique()[:3].tolist() if inv_col(t) in df.columns else None
    diag.append(row)
pd.DataFrame(diag)

,tech,msp_t0,res_cap_t0,max_cap,max_inv
0,pp_biogas,0.00,0.001000,-999.0,[-999.0]
1,pp_biomass,0.00,0.050000,-999.0,[0.0]
2,pp_coal,0.65,4.470295,-999.0,[-999.0]
3,pp_coal_ccs,0.00,0.000000,0.0,[0.0]
4,pp_gas,0.00,2.626988,-999.0,[-999.0]
5,pp_gas_ccs,0.00,0.000000,0.0,[0.0]
6,pp_geothermal,0.00,0.000000,0.0,[0.0]
7,pp_hydropower,0.05,2.643545,-999.0,[-999.0]
8,pp_nuclear,0.00,0.000000,0.0,[0.0]
9,pp_ocean,0.00,0.000000,0.0,[0.0]


Key observations on the original input:
- `pp_nuclear`: `max_capacity = 0` and `max_investment = 0` → **disabled**. Must be enabled for MSP = 0.01 to be feasible.
- `pp_wind`: `max_investment = 0.2` GW/yr → too restrictive for the downstream renewable ramp.
- `pp_biomass`: `max_investment = 0` → cannot grow.

## 3. Apply modifications

In [4]:
df_out = df.copy()

# 3.1 Set the baseline MSP mix (constant across the full horizon)
for t in ALL_PP:
    df_out[msp_col(t)] = MSP_TARGET.get(t, 0.0)

# 3.2 Enable pp_nuclear: drop the max_capacity / max_investment caps
#     (were 0 in the original; set to -999 = no cap)
df_out[cap_col('pp_nuclear')] = -999.0
df_out[inv_col('pp_nuclear')] = -999.0

# 3.3 Loosen investment caps for techs the renewable strategy will need to scale
for t in ['pp_wind', 'pp_biomass', 'pp_solar', 'pp_hydropower']:
    df_out[inv_col(t)] = -999.0

# 3.4 Safety: keep max_elec_production_increase_to_satisfy_msp at -999 to prevent
#     SISEPUEDE from auto-rescaling MSPs and conflicting with NemoMod constraints
for t in ALL_PP:
    if mpi_col(t) in df_out.columns:
        df_out[mpi_col(t)] = -999.0

print('changes applied.')

changes applied.


## 4. Verification

In [5]:
# Σ MSP per year (must be <= 1)
msp_sum = sum(df_out[msp_col(t)] for t in ALL_PP)
assert (msp_sum <= 1.0 + 1e-9).all(), 'Σ MSP > 1 in some year'
print(f'Σ MSP: min={msp_sum.min():.4f}  max={msp_sum.max():.4f}  (expected constant 0.95)')

# Final mix summary
print('\n=== Final baseline mix ===')
for t in ALL_PP:
    v = df_out[msp_col(t)].iloc[0]
    if v > 0:
        print(f'  {t:25s}  {v:.2f}')
print(f'  {"(free for LP)":25s}  {1.0 - msp_sum.iloc[0]:.2f}')

Σ MSP: min=0.9500  max=0.9500  (expected constant 0.95)

=== Final baseline mix ===
  pp_coal                    0.62
  pp_gas                     0.10
  pp_nuclear                 0.01
  pp_oil                     0.04
  pp_solar                   0.04
  pp_wind                    0.14
  (free for LP)              0.05


In [6]:
# Confirm nuclear is enabled and caps are loosened
checks = {
    'pp_nuclear max_capacity':   df_out[cap_col('pp_nuclear')].unique(),
    'pp_nuclear max_investment': df_out[inv_col('pp_nuclear')].unique(),
    'pp_wind max_investment':    df_out[inv_col('pp_wind')].unique(),
    'pp_biomass max_investment': df_out[inv_col('pp_biomass')].unique(),
    'pp_solar max_investment':   df_out[inv_col('pp_solar')].unique(),
}
for k, v in checks.items():
    print(f'{k:35s}  {v}')

pp_nuclear max_capacity              [-999.]
pp_nuclear max_investment            [-999.]
pp_wind max_investment               [-999.]
pp_biomass max_investment            [-999.]
pp_solar max_investment              [-999.]


## 5. Save

In [7]:
df_out.to_csv(OUT, index=False)
print(f'written: {OUT}')
print(f'shape: {df_out.shape}')

written: /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data/df_input_baseline_mix.csv
shape: (56, 2442)
